# Tutorial 01: Hello Tinker

Tinker is a remote GPU service for LLM training and inference. You write training loops in Python on your local machine; Tinker executes the heavy GPU operations (forward passes, backpropagation, sampling) on remote workers.

```
Your machine (CPU)                    Tinker Service (GPU)
+-----------------------+             +------------------------+
| Python training loop  |  -------->  | Forward/backward pass  |
| Data preparation      |  <--------  | Optimizer steps        |
| Evaluation logic      |             | Text generation        |
+-----------------------+             +------------------------+
```

You control the logic. Tinker runs the compute.

## Setup

Install the Tinker SDK and set your API key. Get a key from the [Tinker console](https://tinker-console.thinkingmachines.ai).

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import tinker  # noqa: E402
from tinker import types  # noqa: E402

## The client hierarchy

The entry point to Tinker is the **ServiceClient**. From it, you create specialized clients:

- **SamplingClient** -- generates text from a model (inference)
- **TrainingClient** -- runs forward/backward passes and optimizer steps (training)

Both talk to the same remote GPU workers. Let's start with the ServiceClient.

In [2]:
# Create a ServiceClient. This reads TINKER_API_KEY from your environment.
service_client = tinker.ServiceClient()

# Check what models are available
capabilities = service_client.get_server_capabilities()
print("Available models:")
for model in capabilities.supported_models:
    print(f"  - {model.model_name}")

Available models:
  - deepseek-ai/DeepSeek-V3.1
  - deepseek-ai/DeepSeek-V3.1-Base
  - deepseek-ai/DeepSeek-V3.1:peft:131072
  - moonshotai/Kimi-K2-Thinking
  - moonshotai/Kimi-K2-Thinking:peft:32768:tibo-v2
  - moonshotai/Kimi-K2-Thinking:peft:262144
  - moonshotai/Kimi-K2.5
  - moonshotai/Kimi-K2.5:peft:131072
  - meta-llama/Llama-3.1-70B
  - meta-llama/Llama-3.1-8B
  - meta-llama/Llama-3.1-8B-Instruct
  - meta-llama/Llama-3.2-1B
  - meta-llama/Llama-3.2-3B
  - meta-llama/Llama-3.3-70B-Instruct
  - nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  - nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
  - nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16:peft:262144
  - Qwen/Qwen3-235B-A22B-Instruct-2507
  - Qwen/Qwen3-30B-A3B
  - Qwen/Qwen3-30B-A3B-Base
  - Qwen/Qwen3-30B-A3B-Instruct-2507
  - Qwen/Qwen3-32B
  - Qwen/Qwen3-4B-Instruct-2507
  - Qwen/Qwen3-8B
  - Qwen/Qwen3-8B-Base
  - Qwen/Qwen3-VL-235B-A22B-Instruct
  - Qwen/Qwen3-VL-30B-A3B-Instruct
  - Qwen/Qwen3.5-27B
  - Qwen/Qwen3.5-35B-A3B
  

## Sampling from a model

Let's create a **SamplingClient** to generate text. We will use `Qwen/Qwen3-4B-Instruct-2507`, a compact model that keeps costs low.

The sampling workflow is:
1. Create a `SamplingClient` with a base model name
2. Encode your prompt into tokens using the model's tokenizer
3. Call `sample()` with the prompt and sampling parameters
4. Decode the returned tokens back into text

In [3]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Create a sampling client -- this connects to a remote GPU worker
sampling_client = service_client.create_sampling_client(base_model=MODEL_NAME)

# Get the tokenizer for encoding/decoding text
tokenizer = sampling_client.get_tokenizer()

In [4]:
# Encode a prompt into tokens
prompt_text = "The three largest cities in the world by population are"
prompt = types.ModelInput.from_ints(tokenizer.encode(prompt_text))

# Sample a completion
params = types.SamplingParams(max_tokens=50, temperature=0.7, stop=["\n"])
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=1)

# .sample() returns a future immediately -- call .result() to wait for the GPU response
result = future.result()

# Decode and print
completion_tokens = result.sequences[0].tokens
print(prompt_text + tokenizer.decode(completion_tokens))

The three largest cities in the world by population are: Tokyo, Delhi, and Shanghai. Tokyo is 10 times larger than Delhi, and Delhi is 3 times larger than Shanghai. If Shanghai has a population of 2 million, what is the total population of the three cities?




## Inspecting the response

The `sample()` call returns a `SampleResponse` containing a list of `SampledSequence` objects. Each sequence has:
- `tokens` -- the generated token IDs
- `logprobs` -- log probability of each generated token (if requested)
- `stop_reason` -- why generation stopped (e.g., hit max tokens, hit a stop string)

In [5]:
seq = result.sequences[0]

print(f"Stop reason:    {seq.stop_reason}")
print(f"Tokens generated: {len(seq.tokens)}")
print(f"Token IDs:      {seq.tokens[:10]}...")  # first 10
print(f"Log probs:      {seq.logprobs}")

Stop reason:    stop
Tokens generated: 48
Token IDs:      [25, 26194, 11, 21996, 11, 323, 37047, 13, 26194, 374]...
Log probs:      [-2.36488676071167, -0.48236992955207825, -0.02776242606341839, -0.17256887257099152, -9.297892393078655e-05, -1.1920922133867862e-06, -0.05663525313138962, -0.004779462236911058, -2.5594124794006348, -1.7229485511779785, -0.31663620471954346, -0.7455677390098572, -0.7929264903068542, -2.0899343490600586, -0.4618220925331116, -0.009594282135367393, -0.07574518024921417, -0.3136867582798004, -0.00043561504571698606, -0.5341041088104248, -2.4199192921514623e-05, -0.004535031970590353, -1.313751459121704, -0.002300118561834097, -0.010874061845242977, -1.1920928244535389e-07, -2.3841855067985307e-07, -2.372236667724792e-05, -0.022631576284766197, -1.3918805122375488, -0.009919043630361557, -0.009588733315467834, -1.7881377516459906e-06, -1.1920928244535389e-07, -0.0008694920688867569, -0.6940340399742126, -2.9721293449401855, -0.000662822334561497, -0.04504619

You can also generate multiple samples at once by setting `num_samples`. Each sample is an independent completion from the same prompt.

In [6]:
# Generate 3 independent completions
result = sampling_client.sample(
    prompt=prompt,
    sampling_params=types.SamplingParams(max_tokens=50, temperature=0.9, stop=["\n"]),
    num_samples=3,
).result()

for i, seq in enumerate(result.sequences):
    text = tokenizer.decode(seq.tokens)
    print(f"Sample {i}: {text}")

Sample 0:  New York City, Tokyo, and London. The population of New York City is 8.8 million, Tokyo is 14.0 million, and London is 9.0 million. What is the total population of these three cities?

Sample 1:  Tokyo, Osaka, and Nagoya. The population of Tokyo is 10% greater than the population of Osaka, and the population of Osaka is 20% less than the population of Nagoya. If the population of Nagoya is 
Sample 2:  Tokyo, Seoul, and Shanghai. If Tokyo has a population of 10 million, Seoul has a population of 10 million, and Shanghai has a population of 25 million, what is the total population of these three cities?



## What about training?

So far we have only done inference. The real power of Tinker is **training** -- running forward/backward passes and optimizer steps on remote GPUs while you control the training loop locally.

The workflow looks like this:

1. Create a **TrainingClient** with `service_client.create_lora_training_client()`
2. Prepare training data as `Datum` objects (input tokens + loss targets)
3. Call `training_client.forward_backward()` to compute gradients
4. Call `training_client.optim_step()` to update weights
5. Save weights and create a **SamplingClient** to evaluate the trained model

We will walk through this in the next tutorial.

## Next steps

- **[Tutorial 02: First SFT](./02_first_sft.ipynb)** -- Train a model with supervised fine-tuning
- **[Getting Started Guide](https://docs.thinkingmachines.ai/training-sampling)** -- Full walkthrough of training and sampling
- **[Available Models](https://docs.thinkingmachines.ai/model-lineup)** -- All supported models and their characteristics